# 🔍 Jigsaw Toxic Comment Classification - Comprehensive EDA & ML Analysis

This notebook provides a complete exploratory data analysis (EDA), data preprocessing, feature engineering, and machine learning pipeline for the **Jigsaw Toxic Comment Classification Challenge**.

## 🎯 Objective
Build a multi-headed model that's capable of detecting different types of toxicity like threats, obscenity, insults, and identity-based hate in online comments.

## 📊 Dataset
The dataset contains comments from Wikipedia's talk page edits, labeled by human raters for toxic behavior. The types of toxicity are:
- **toxic**: General toxicity
- **severe_toxic**: Severe toxicity
- **obscene**: Obscene language
- **threat**: Threats
- **insult**: Insults
- **identity_hate**: Identity-based hate

## Table of Contents
1. [Data Insights](#1-data-insights)
2. [Numerical Feature Histograms](#2-numerical-feature-histograms)
3. [Categorical Feature Bar Charts](#3-categorical-feature-bar-charts)
4. [Categorical vs Categorical Analysis](#4-categorical-vs-categorical-analysis)
5. [Categorical vs Numerical Analysis](#5-categorical-vs-numerical-analysis)
6. [Numerical vs Numerical Pair Plots](#6-numerical-vs-numerical-pair-plots)
7. [Missing Data Handling](#7-missing-data-handling)
8. [Stratified Sampling](#8-stratified-sampling)
9. [One-Hot Encoding](#9-one-hot-encoding)
10. [Normalization/Standardization](#10-normalizationstandardization)
11. [Balancing Techniques](#11-balancing-techniques)
12. [Correlation Matrix Analysis](#12-correlation-matrix-analysis)
13. [Outlier Removal](#13-outlier-removal)
14. [Machine Learning Models Comparison](#14-machine-learning-models-comparison)


## Setup and Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import sys

# Statistical libraries
from scipy import stats
from scipy.stats import chi2_contingency, pearsonr, spearmanr

# Machine Learning libraries
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Imbalanced learning libraries
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek

# Text processing utilities
import nltk
from collections import Counter
# Additional imports for text processing and machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import re
import string

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✅ All libraries imported successfully!')

# Kaggle-specific setup
try:
    # Download NLTK data if needed
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    print('📚 NLTK data downloaded successfully!')
except Exception as e:
    print(f'⚠️ NLTK download warning: {e}')

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print('🎯 Environment setup complete!')

## 1. Data Insights

Let's start by loading the dataset and performing basic data exploration using df.info() and other descriptive statistics.

In [ ]:
# Load Jigsaw Toxic Comment Classification Dataset
import os

# Kaggle input directory
input_dir = '/kaggle/input/jigsaw-toxic-comment-classification-challenge'

# Check if running on Kaggle or locally
if os.path.exists(input_dir):
    # Running on Kaggle
    train_path = os.path.join(input_dir, 'train.csv')
    test_path = os.path.join(input_dir, 'test.csv')
    print('📊 Loading data from Kaggle input directory...')
else:
    # Running locally - adjust paths as needed
    train_path = 'data/train.csv'
    test_path = 'data/test.csv'
    print('📊 Loading data from local directory...')

# Load the datasets
try:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Combine for comprehensive EDA (using sample for performance)
    sample_size = 10000  # Adjust based on computational resources
    if len(train_df) > sample_size:
        train_sample = train_df.sample(n=sample_size, random_state=42)
    else:
        train_sample = train_df
    
    # Use training data for EDA (test data doesn't have labels)
    df = train_sample.copy()
    
    print(f'✅ Successfully loaded {len(df)} samples for EDA')
    print(f'📈 Training data shape: {train_df.shape}')
    print(f'📈 Test data shape: {test_df.shape}')
    
except FileNotFoundError as e:
    print(f'❌ Error loading data: {e}')
    print('Please ensure the dataset is available in the correct directory')
    # Create sample data for demonstration
    print('Creating sample data for demonstration...')
    df = pd.DataFrame({
        'id': range(1000),
        'comment_text': ['Sample comment ' + str(i) for i in range(1000)],
        'toxic': np.random.choice([0, 1], 1000),
        'severe_toxic': np.random.choice([0, 1], 1000),
        'obscene': np.random.choice([0, 1], 1000),
        'threat': np.random.choice([0, 1], 1000),
        'insult': np.random.choice([0, 1], 1000),
        'identity_hate': np.random.choice([0, 1], 1000)
    })

# Standardize column names for analysis
if 'comment_text' in df.columns:
    df['text'] = df['comment_text']  # Standardize text column name

# Create a binary label for overall toxicity
toxic_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
available_toxic_cols = [col for col in toxic_columns if col in df.columns]

if available_toxic_cols:
    df['label'] = (df[available_toxic_cols].sum(axis=1) > 0).astype(int)
    print(f'📊 Created binary toxicity label from columns: {available_toxic_cols}')

print(f'🎯 Final dataset shape for EDA: {df.shape}')
# Display basic information about the loaded dataset
print('\n' + '='*50)
print('DATASET OVERVIEW')
print('='*50)
print(f'Dataset columns: {list(df.columns)}')
print(f'Dataset shape: {df.shape}')

# Check for text and label columns
text_cols = [col for col in df.columns if 'text' in col.lower() or 'comment' in col.lower()]
label_cols = [col for col in df.columns if col in ['label', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]

print(f'Text columns found: {text_cols}')
print(f'Label columns found: {label_cols}')

In [ ]:
# Basic Data Insights using df.info()
print('=' * 60)
print('BASIC DATA INSIGHTS')
print('=' * 60)

print('1. Dataset Information:')
print('-' * 30)
df.info()

print('\n2. Dataset Shape:')
print('-' * 30)
print(f'Rows: {df.shape[0]:}')
print(f'Columns: {df.shape[1]:}')

print('\n3. Column Names and Types:')
print('-' * 30)
for col in df.columns:
    print(f'{col}: {df[col].dtype}')

print('\n4. First 5 Rows:')
print('-' * 30)
display(df.head())

print('\n5. Last 5 Rows:')
print('-' * 30)
display(df.tail())

print('\n6. Basic Statistics:')
print('-' * 30)
display(df.describe(include='all'))


In [ ]:
# Add derived features for comprehensive analysis
print('Adding derived features for comprehensive analysis...')

# Text-based features
if 'text' in df.columns:
    df['text_length'] = df['text'].astype(str).apply(len)
    df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
    df['char_count'] = df['text'].astype(str).apply(lambda x: len(x.replace(' ', '')))
    df['sentence_count'] = df['text'].astype(str).apply(lambda x: len(x.split('.')))
    df['avg_word_length'] = df['char_count'] / df['word_count'].replace(0, 1)
    df['punctuation_count'] = df['text'].astype(str).apply(lambda x: sum(1 for c in x if c in '.,!?;:'))
    df['uppercase_count'] = df['text'].astype(str).apply(lambda x: sum(1 for c in x if c.isupper()))
    df['digit_count'] = df['text'].astype(str).apply(lambda x: sum(1 for c in x if c.isdigit()))

# Label encoding for categorical analysis
if 'label' in df.columns:
    df['label_encoded'] = LabelEncoder().fit_transform(df['label'])

print('✅ Derived features added successfully!')
print(f'Updated dataset shape: {df.shape}')
print(f'New columns: {[col for col in df.columns if col.endswith((\"_count\", \"_length\", \"_encoded\"))]}')

## 2. Numerical Feature Histograms

Creating individual histograms for numerical features and identifying the 5 most essential plots.

In [ ]:
# Identify numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns found: {numerical_cols}')

# Create histograms for all numerical features
n_cols = len(numerical_cols)
if n_cols > 0:
    n_rows = (n_cols + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 6 * n_rows))
    axes = axes.flatten() if n_cols > 1 else [axes]
    
    for i, col in enumerate(numerical_cols):
        if i < len(axes):
            axes[i].hist(df[col].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black')
            axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
            
            # Add statistics
            mean_val = df[col].mean()
            median_val = df[col].median()
            axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
            axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
            axes[i].legend()
    
    # Hide empty subplots
    for i in range(n_cols, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Individual Histograms for Numerical Features', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
else:
    print('No numerical columns found for histogram analysis.')


In [ ]:
# Identify and explain the 5 most essential numerical plots
print('=' * 60)
print('5 MOST ESSENTIAL NUMERICAL HISTOGRAMS')
print('=' * 60)

# Calculate importance metrics for each numerical column
importance_metrics = {}

for col in numerical_cols:
    if col in df.columns:
        # Calculate various importance metrics
        variance = df[col].var()
        skewness = abs(df[col].skew())
        kurtosis = abs(df[col].kurtosis())
        range_val = df[col].max() - df[col].min()
        cv = df[col].std() / df[col].mean() if df[col].mean() != 0 else 0
        
        # Composite importance score
        importance_score = variance * 0.3 + skewness * 0.2 + kurtosis * 0.2 + range_val * 0.2 + cv * 0.1
        importance_metrics[col] = {
            'importance_score': importance_score,
            'variance': variance,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'range': range_val,
            'cv': cv
        }

# Sort by importance and select top 5
top_5_numerical = sorted(importance_metrics.items(), key=lambda x: x[1]['importance_score'], reverse=True)[:5]

# Create detailed plots for top 5
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, (col, metrics) in enumerate(top_5_numerical):
    if i < 5:
        # Histogram with KDE
        axes[i].hist(df[col].dropna(), bins=30, alpha=0.7, density=True, color='lightblue', edgecolor='black')
        
        # Add KDE curve
        try:
            df[col].dropna().plot.kde(ax=axes[i], color='red', linewidth=2)
        except:
            pass
        
        axes[i].set_title(f'{col}\n(Importance Score: {metrics["importance_score"]:.2f})', 
                         fontsize=12, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Density')
        axes[i].grid(True, alpha=0.3)
        
        # Add statistics text
        stats_text = f"Mean: {df[col].mean():.2f}\nStd: {df[col].std():.2f}\nSkew: {df[col].skew():.2f}"
        axes[i].text(0.02, 0.98, stats_text, transform=axes[i].transAxes, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Hide the 6th subplot
axes[5].set_visible(False)

plt.tight_layout()
plt.suptitle('Top 5 Most Essential Numerical Feature Histograms', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Print explanations
print('\nEXPLANATIONS FOR TOP 5 NUMERICAL HISTOGRAMS:')
print('-' * 50)

explanations = {
    'text_length': 'Text length distribution shows the variety in content size, helping identify outliers and optimal preprocessing strategies.',
    'word_count': 'Word count distribution reveals content complexity and can indicate different text types or quality levels.',
    'punctuation_count': 'Punctuation usage patterns can indicate writing style, formality, and emotional content.',
    'avg_word_length': 'Average word length distribution shows vocabulary complexity and reading difficulty levels.',
    'label_encoded': 'Label distribution shows class balance, crucial for understanding potential bias and sampling needs.'
}

for i, (col, metrics) in enumerate(top_5_numerical):
    print(f'{i+1}. {col.upper()}:')
    print(f'   Importance Score: {metrics["importance_score"]:.3f}')
    print(f'   Explanation: {explanations.get(col, "This feature shows significant statistical variation and distribution characteristics important for model performance.")}')
    print(f'   Key Stats: Variance={metrics["variance"]:.2f}, Skewness={metrics["skewness"]:.2f}, Range={metrics["range"]:.2f}')
    print()


## 3. Categorical Feature Bar Charts

Creating individual bar charts for categorical features and identifying the 5 most essential plots.

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Add some derived categorical features
if 'text_length' in df.columns:
    df['text_length_category'] = pd.cut(df['text_length'], 
                                       bins=[0, 50, 150, 300, float('inf')], 
                                       labels=['Very Short', 'Short', 'Medium', 'Long'])
    categorical_cols.append('text_length_category')

if 'word_count' in df.columns:
    df['word_count_category'] = pd.cut(df['word_count'], 
                                      bins=[0, 10, 30, 60, float('inf')], 
                                      labels=['Few Words', 'Some Words', 'Many Words', 'Very Many Words'])
    categorical_cols.append('word_count_category')

if 'punctuation_count' in df.columns:
    df['punctuation_category'] = pd.cut(df['punctuation_count'], 
                                       bins=[-1, 0, 2, 5, float('inf')], 
                                       labels=['No Punctuation', 'Low', 'Medium', 'High'])
    categorical_cols.append('punctuation_category')

print(f'Categorical columns found: {categorical_cols}')

# Create bar charts for all categorical features
n_cols = len(categorical_cols)
if n_cols > 0:
    n_rows = (n_cols + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 6 * n_rows))
    axes = axes.flatten() if n_cols > 1 else [axes]
    
    for i, col in enumerate(categorical_cols):
        if i < len(axes) and col in df.columns:
            value_counts = df[col].value_counts()
            
            # Create bar plot
            bars = axes[i].bar(range(len(value_counts)), value_counts.values, 
                              color=plt.cm.Set3(np.linspace(0, 1, len(value_counts))))
            
            axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Count')
            axes[i].set_xticks(range(len(value_counts)))
            axes[i].set_xticklabels(value_counts.index, rotation=45, ha='right')
            axes[i].grid(True, alpha=0.3)
            
            # Add value labels on bars
            for bar, value in zip(bars, value_counts.values):
                axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(value_counts.values)*0.01,
                           f'{value}', ha='center', va='bottom', fontsize=10)
    
    # Hide empty subplots
    for i in range(n_cols, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Individual Bar Charts for Categorical Features', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
else:
    print('No categorical columns found for bar chart analysis.')


In [ ]:
# Identify and explain the 5 most essential categorical plots
print('=' * 60)
print('5 MOST ESSENTIAL CATEGORICAL BAR CHARTS')
print('=' * 60)

# Calculate importance metrics for each categorical column
categorical_importance = {}

for col in categorical_cols:
    if col in df.columns:
        value_counts = df[col].value_counts()
        
        # Calculate importance metrics
        n_categories = len(value_counts)
        entropy = -sum((p := value_counts / len(df)) * np.log2(p + 1e-10))
        balance_score = 1 - (value_counts.max() - value_counts.min()) / value_counts.sum()
        coverage = (value_counts > 0).sum() / len(value_counts) if len(value_counts) > 0 else 0
        
        # Composite importance score
        importance_score = entropy * 0.4 + balance_score * 0.3 + n_categories * 0.2 + coverage * 0.1
        
        categorical_importance[col] = {
            'importance_score': importance_score,
            'n_categories': n_categories,
            'entropy': entropy,
            'balance_score': balance_score,
            'coverage': coverage
        }

# Sort by importance and select top 5
top_5_categorical = sorted(categorical_importance.items(), key=lambda x: x[1]['importance_score'], reverse=True)[:5]

# Create detailed plots for top 5
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, (col, metrics) in enumerate(top_5_categorical):
    if i < 5 and col in df.columns:
        value_counts = df[col].value_counts()
        
        # Create enhanced bar plot
        bars = axes[i].bar(range(len(value_counts)), value_counts.values,
                          color=plt.cm.viridis(np.linspace(0, 1, len(value_counts))))
        
        axes[i].set_title(f'{col}\n(Importance Score: {metrics["importance_score"]:.2f})', 
                         fontsize=12, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')
        axes[i].set_xticks(range(len(value_counts)))
        axes[i].set_xticklabels(value_counts.index, rotation=45, ha='right')
        axes[i].grid(True, alpha=0.3)
        
        # Add percentage labels
        total = value_counts.sum()
        for bar, value in zip(bars, value_counts.values):
            percentage = (value / total) * 100
            axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(value_counts.values)*0.01,
                       f'{value}\n({percentage:.1f}%)', ha='center', va='bottom', fontsize=9)

# Hide the 6th subplot
axes[5].set_visible(False)

plt.tight_layout()
plt.suptitle('Top 5 Most Essential Categorical Feature Bar Charts', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Print explanations
print('\nEXPLANATIONS FOR TOP 5 CATEGORICAL BAR CHARTS:')
print('-' * 50)

explanations = {
    'label': 'Target variable distribution is crucial for understanding class balance and potential bias in the dataset.',
    'text_length_category': 'Text length categories help identify content patterns and optimal preprocessing strategies.',
    'word_count_category': 'Word count categories reveal content complexity and can guide feature engineering decisions.',
    'punctuation_category': 'Punctuation usage patterns indicate writing style and emotional content characteristics.',
    'dataset_source': 'Dataset source distribution shows data diversity and potential domain-specific patterns.'
}

for i, (col, metrics) in enumerate(top_5_categorical):
    print(f'{i+1}. {col.upper()}:')
    print(f'   Importance Score: {metrics["importance_score"]:.3f}')
    print(f'   Explanation: {explanations.get(col, "This categorical feature shows significant diversity and distribution patterns important for understanding data structure.")}')
    print(f'   Key Stats: Categories={metrics["n_categories"]}, Entropy={metrics["entropy"]:.2f}, Balance={metrics["balance_score"]:.2f}')
    print()


## 4. Categorical vs Categorical Analysis

Creating bar plots to analyze relationships between categorical variables.

In [ ]:
# Categorical vs Categorical Analysis
print('=' * 60)
print('CATEGORICAL vs CATEGORICAL ANALYSIS')
print('=' * 60)

# Find pairs of categorical variables for analysis
categorical_pairs = []

# Prioritize pairs that include the target variable
if 'label' in categorical_cols:
    for col in categorical_cols:
        if col != 'label' and col in df.columns:
            categorical_pairs.append(('label', col))

# Add other interesting pairs
other_pairs = [
    ('text_length_category', 'word_count_category'),
    ('text_length_category', 'punctuation_category'),
    ('word_count_category', 'punctuation_category')
]

for pair in other_pairs:
    if pair[0] in df.columns and pair[1] in df.columns:
        categorical_pairs.append(pair)

# Limit to top 6 pairs for visualization
categorical_pairs = categorical_pairs[:6]

if categorical_pairs:
    n_pairs = len(categorical_pairs)
    n_rows = (n_pairs + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(20, 6 * n_rows))
    axes = axes.flatten() if n_pairs > 1 else [axes]
    
    for i, (cat1, cat2) in enumerate(categorical_pairs):
        if i < len(axes):
            # Create crosstab
            crosstab = pd.crosstab(df[cat1], df[cat2])
            
            # Create stacked bar plot
            crosstab.plot(kind='bar', stacked=True, ax=axes[i], 
                         colormap='Set3', alpha=0.8)
            
            axes[i].set_title(f'{cat1} vs {cat2}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(cat1)
            axes[i].set_ylabel('Count')
            axes[i].legend(title=cat2, bbox_to_anchor=(1.05, 1), loc='upper left')
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].grid(True, alpha=0.3)
            
            # Calculate and display chi-square test
            try:
                chi2, p_value, dof, expected = chi2_contingency(crosstab)
                axes[i].text(0.02, 0.98, f'χ² = {chi2:.2f}\np = {p_value:.4f}', 
                           transform=axes[i].transAxes, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            except:
                pass
    
    # Hide empty subplots
    for i in range(n_pairs, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Categorical vs Categorical Relationships', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Print statistical analysis
    print('\nSTATISTICAL ANALYSIS OF CATEGORICAL RELATIONSHIPS:')
    print('-' * 50)
    
    for cat1, cat2 in categorical_pairs:
        crosstab = pd.crosstab(df[cat1], df[cat2])
        try:
            chi2, p_value, dof, expected = chi2_contingency(crosstab)
            print(f'{cat1} vs {cat2}:')
            print(f'  Chi-square statistic: {chi2:.4f}')
            print(f'  P-value: {p_value:.4f}')
            print(f'  Degrees of freedom: {dof}')
            print(f'  Relationship: {"Significant" if p_value < 0.05 else "Not significant"} (α = 0.05)')
            print()
        except Exception as e:
            print(f'{cat1} vs {cat2}: Unable to calculate chi-square test ({e})')
            print()
else:
    print('No suitable categorical pairs found for analysis.')


## 5. Categorical vs Numerical Analysis

Creating plots to analyze relationships between categorical and numerical variables with different colors.

In [ ]:
# Categorical vs Numerical Analysis
print('=' * 60)
print('CATEGORICAL vs NUMERICAL ANALYSIS')
print('=' * 60)

# Find interesting categorical vs numerical pairs
cat_num_pairs = []

# Prioritize pairs with target variable
if 'label' in categorical_cols:
    for num_col in numerical_cols:
        if num_col != 'label_encoded':
            cat_num_pairs.append(('label', num_col))

# Add other interesting pairs
other_cat_num_pairs = [
    ('text_length_category', 'word_count'),
    ('word_count_category', 'text_length'),
    ('punctuation_category', 'text_length'),
    ('text_length_category', 'punctuation_count')
]

for pair in other_cat_num_pairs:
    if pair[0] in df.columns and pair[1] in df.columns:
        cat_num_pairs.append(pair)

# Limit to top 6 pairs
cat_num_pairs = cat_num_pairs[:6]

if cat_num_pairs:
    n_pairs = len(cat_num_pairs)
    n_rows = (n_pairs + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(20, 6 * n_rows))
    axes = axes.flatten() if n_pairs > 1 else [axes]
    
    for i, (cat_col, num_col) in enumerate(cat_num_pairs):
        if i < len(axes):
            # Create box plot with different colors
            unique_categories = df[cat_col].dropna().unique()
            colors = plt.cm.Set2(np.linspace(0, 1, len(unique_categories)))
            
            box_data = [df[df[cat_col] == cat][num_col].dropna() for cat in unique_categories]
            
            bp = axes[i].boxplot(box_data, labels=unique_categories, patch_artist=True)
            
            # Color the boxes
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            axes[i].set_title(f'{cat_col} vs {num_col}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(cat_col)
            axes[i].set_ylabel(num_col)
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].grid(True, alpha=0.3)
            
            # Add statistical information
            try:
                # Perform ANOVA test
                groups = [df[df[cat_col] == cat][num_col].dropna() for cat in unique_categories]
                f_stat, p_value = stats.f_oneway(*groups)
                axes[i].text(0.02, 0.98, f'F = {f_stat:.2f}\np = {p_value:.4f}', 
                           transform=axes[i].transAxes, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            except:
                pass
    
    # Hide empty subplots
    for i in range(n_pairs, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Categorical vs Numerical Relationships (Box Plots)', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Create violin plots for better distribution visualization
    fig, axes = plt.subplots(n_rows, 3, figsize=(20, 6 * n_rows))
    axes = axes.flatten() if n_pairs > 1 else [axes]
    
    for i, (cat_col, num_col) in enumerate(cat_num_pairs):
        if i < len(axes):
            # Create violin plot
            sns.violinplot(data=df, x=cat_col, y=num_col, ax=axes[i], palette='Set2')
            
            axes[i].set_title(f'{cat_col} vs {num_col} (Distribution)', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(cat_col)
            axes[i].set_ylabel(num_col)
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].grid(True, alpha=0.3)
    
    # Hide empty subplots
    for i in range(n_pairs, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle('Categorical vs Numerical Relationships (Violin Plots)', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Print statistical analysis
    print('\nSTATISTICAL ANALYSIS OF CATEGORICAL vs NUMERICAL RELATIONSHIPS:')
    print('-' * 60)
    
    for cat_col, num_col in cat_num_pairs:
        print(f'{cat_col} vs {num_col}:')
        
        # Group statistics
        group_stats = df.groupby(cat_col)[num_col].agg(['count', 'mean', 'std', 'min', 'max'])
        print('  Group Statistics:')
        print(group_stats.round(2))
        
        # ANOVA test
        try:
            unique_categories = df[cat_col].dropna().unique()
            groups = [df[df[cat_col] == cat][num_col].dropna() for cat in unique_categories]
            f_stat, p_value = stats.f_oneway(*groups)
            print(f'  ANOVA Test:')
            print(f'    F-statistic: {f_stat:.4f}')
            print(f'    P-value: {p_value:.4f}')
            print(f'    Significant difference: {"Yes" if p_value < 0.05 else "No"} (α = 0.05)')
        except Exception as e:
            print(f'  ANOVA Test: Unable to calculate ({e})')
        
        print('-' * 40)
        print()
else:
    print('No suitable categorical vs numerical pairs found for analysis.')


## 6. Numerical vs Numerical Pair Plots

Creating pair plots to analyze relationships between numerical variables.

In [ ]:
# Numerical vs Numerical Pair Plots
print('=' * 60)
print('NUMERICAL vs NUMERICAL PAIR PLOTS')
print('=' * 60)

# Select most relevant numerical columns for pair plot
# Limit to avoid overcrowded plots
selected_numerical = []

# Prioritize derived text features
priority_cols = ['text_length', 'word_count', 'char_count', 'avg_word_length', 
                'punctuation_count', 'uppercase_count', 'digit_count']

for col in priority_cols:
    if col in numerical_cols and col in df.columns:
        selected_numerical.append(col)

# Add other numerical columns if we have less than 6
for col in numerical_cols:
    if col not in selected_numerical and len(selected_numerical) < 6:
        selected_numerical.append(col)

# Limit to maximum 6 variables for readability
selected_numerical = selected_numerical[:6]

print(f'Selected numerical columns for pair plot: {selected_numerical}')

if len(selected_numerical) >= 2:
    # Create subset dataframe
    pair_df = df[selected_numerical + (['label'] if 'label' in df.columns else [])].copy()
    
    # Remove any infinite or extremely large values
    for col in selected_numerical:
        pair_df[col] = pair_df[col].replace([np.inf, -np.inf], np.nan)
        # Cap extreme outliers at 99th percentile
        q99 = pair_df[col].quantile(0.99)
        pair_df[col] = pair_df[col].clip(upper=q99)
    
    # Drop rows with missing values
    pair_df = pair_df.dropna()
    
    print(f'Pair plot data shape: {pair_df.shape}')
    
    # Create pair plot
    if 'label' in pair_df.columns:
        # Pair plot with hue based on target variable
        g = sns.pairplot(pair_df, hue='label', diag_kind='hist', 
                        plot_kws={'alpha': 0.6}, diag_kws={'alpha': 0.7})
        g.fig.suptitle('Numerical Feature Pair Plot (Colored by Target)', 
                      fontsize=16, fontweight='bold', y=1.02)
    else:
        # Simple pair plot without hue
        g = sns.pairplot(pair_df, diag_kind='hist', 
                        plot_kws={'alpha': 0.6}, diag_kws={'alpha': 0.7})
        g.fig.suptitle('Numerical Feature Pair Plot', 
                      fontsize=16, fontweight='bold', y=1.02)
    
    plt.show()
    
    # Calculate and display correlation matrix
    print('\nCORRELATION MATRIX FOR SELECTED NUMERICAL FEATURES:')
    print('-' * 50)
    
    correlation_matrix = pair_df[selected_numerical].corr()
    
    # Create correlation heatmap
    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', 
                center=0, square=True, fmt='.3f', cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix - Numerical Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print correlation analysis
    print('\nSTRONG CORRELATIONS (|r| > 0.5):')
    print('-' * 30)
    
    strong_correlations = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.5:
                strong_correlations.append((
                    correlation_matrix.columns[i], 
                    correlation_matrix.columns[j], 
                    corr_val
                ))
    
    if strong_correlations:
        for var1, var2, corr in sorted(strong_correlations, key=lambda x: abs(x[2]), reverse=True):
            print(f'{var1} ↔ {var2}: r = {corr:.3f}')
    else:
        print('No strong correlations found (|r| > 0.5)')
    
    print('\nCORRELATION INTERPRETATION:')
    print('-' * 25)
    print('• |r| > 0.7: Strong correlation')
    print('• 0.5 < |r| ≤ 0.7: Moderate correlation')
    print('• 0.3 < |r| ≤ 0.5: Weak correlation')
    print('• |r| ≤ 0.3: Very weak/no correlation')

else:
    print('Insufficient numerical columns for pair plot analysis.')


## 7. Missing Data Handling

Analyzing and handling missing data in the dataset.

In [ ]:
# Missing Data Analysis
print('=' * 60)
print('MISSING DATA ANALYSIS')
print('=' * 60)

# Calculate missing data statistics
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_data.index,
    'Missing_Count': missing_data.values,
    'Missing_Percentage': missing_percent.values
})

# Sort by missing percentage
missing_df = missing_df.sort_values('Missing_Percentage', ascending=False)
missing_df = missing_df[missing_df['Missing_Count'] > 0]

print('MISSING DATA SUMMARY:')
print('-' * 30)
if len(missing_df) > 0:
    display(missing_df)
else:
    print('✅ No missing data found in the dataset!')

# Visualize missing data
if len(missing_df) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar plot of missing data
    ax1.bar(range(len(missing_df)), missing_df['Missing_Percentage'], 
           color='salmon', alpha=0.7)
    ax1.set_title('Missing Data Percentage by Column', fontweight='bold')
    ax1.set_xlabel('Columns')
    ax1.set_ylabel('Missing Percentage (%)')
    ax1.set_xticks(range(len(missing_df)))
    ax1.set_xticklabels(missing_df['Column'], rotation=45, ha='right')
    ax1.grid(True, alpha=0.3)
    
    # Missing data heatmap
    missing_matrix = df[missing_df['Column']].isnull()
    sns.heatmap(missing_matrix.T, cbar=True, ax=ax2, cmap='viridis')
    ax2.set_title('Missing Data Pattern', fontweight='bold')
    ax2.set_xlabel('Samples')
    ax2.set_ylabel('Features')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Missing Data Handling Strategies
print('\nMISSING DATA HANDLING STRATEGIES:')
print('-' * 40)

# Create a copy for handling missing data
df_cleaned = df.copy()

# Strategy 1: Remove columns with >50% missing data
high_missing_cols = missing_df[missing_df['Missing_Percentage'] > 50]['Column'].tolist()
if high_missing_cols:
    print(f'Removing columns with >50% missing data: {high_missing_cols}')
    df_cleaned = df_cleaned.drop(columns=high_missing_cols)

# Strategy 2: Handle numerical missing data
numerical_missing = missing_df[
    (missing_df['Missing_Percentage'] <= 50) & 
    (missing_df['Column'].isin(numerical_cols))
]

for col in numerical_missing['Column']:
    if col in df_cleaned.columns:
        # Use median for numerical data
        median_val = df_cleaned[col].median()
        df_cleaned[col].fillna(median_val, inplace=True)
        print(f'Filled {col} missing values with median: {median_val:.2f}')

# Strategy 3: Handle categorical missing data
categorical_missing = missing_df[
    (missing_df['Missing_Percentage'] <= 50) & 
    (missing_df['Column'].isin(categorical_cols))
]

for col in categorical_missing['Column']:
    if col in df_cleaned.columns:
        # Use mode for categorical data
        mode_val = df_cleaned[col].mode()[0] if not df_cleaned[col].mode().empty else 'Unknown'
        df_cleaned[col].fillna(mode_val, inplace=True)
        print(f'Filled {col} missing values with mode: {mode_val}')

# Strategy 4: Handle text missing data
if 'text' in df_cleaned.columns:
    text_missing = df_cleaned['text'].isnull().sum()
    if text_missing > 0:
        df_cleaned['text'].fillna('', inplace=True)
        print(f'Filled {text_missing} missing text values with empty string')

# Final missing data check
final_missing = df_cleaned.isnull().sum().sum()
print(f'\n✅ Final missing data count: {final_missing}')
print(f'Dataset shape after cleaning: {df_cleaned.shape}')

## 8. Stratified Sampling

Implementing stratified sampling for imbalanced classes.

In [ ]:
# Stratified Sampling Analysis
print('=' * 60)
print('STRATIFIED SAMPLING FOR IMBALANCED CLASSES')
print('=' * 60)

# Check class distribution
if 'label' in df_cleaned.columns:
    class_distribution = df_cleaned['label'].value_counts()
    class_percentages = df_cleaned['label'].value_counts(normalize=True) * 100
    
    print('ORIGINAL CLASS DISTRIBUTION:')
    print('-' * 30)
    for label, count in class_distribution.items():
        percentage = class_percentages[label]
        print(f'{label}: {count:} samples ({percentage:.2f}%)')
    
    # Calculate imbalance ratio
    max_class = class_distribution.max()
    min_class = class_distribution.min()
    imbalance_ratio = max_class / min_class
    
    print(f'\nImbalance Ratio: {imbalance_ratio:.2f}:1')
    
    if imbalance_ratio > 1.5:
        print('⚠️  Dataset is imbalanced - stratified sampling recommended')
    else:
        print('✅ Dataset is relatively balanced')
    
    # Perform stratified sampling
    if 'text' in df_cleaned.columns:
        X = df_cleaned['text']
        y = df_cleaned['label']
        
        # Stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
        
        print(f'\nTraining set: {len(X_train):} samples')
        print(f'Test set: {len(X_test):} samples')
        
        # Store for later use
        stratified_data = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test
        }
        
        print('✅ Stratified sampling completed!')
    else:
        print('❌ Text column not found')
else:
    print('❌ Label column not found')

## 9. One-Hot Encoding

Applying one-hot encoding for categorical features.

In [ ]:
# One-Hot Encoding for Categorical Features
print('=' * 60)
print('ONE-HOT ENCODING FOR CATEGORICAL FEATURES')
print('=' * 60)

# Identify categorical columns for encoding (exclude target)
categorical_for_encoding = [col for col in categorical_cols 
                           if col in df_cleaned.columns and col != 'label']

print(f'Categorical columns for encoding: {categorical_for_encoding}')

if categorical_for_encoding:
    # Create a copy for encoding
    df_encoded = df_cleaned.copy()
    encoded_features = []
    
    for col in categorical_for_encoding:
        cardinality = df_encoded[col].nunique()
        print(f'\n{col}: {cardinality} unique values')
        
        if cardinality <= 10:
            # One-hot encode
            encoded_cols = pd.get_dummies(df_encoded[col], prefix=col)
            df_encoded = pd.concat([df_encoded, encoded_cols], axis=1)
            df_encoded.drop(columns=[col], inplace=True)
            encoded_features.extend(encoded_cols.columns.tolist())
            print(f'  ✅ Encoded into {len(encoded_cols.columns)} features')
        else:
            print(f'  ⚠️  Skipped (too many categories: {cardinality})')
    
    print(f'\nTotal encoded features: {len(encoded_features)}')
    print(f'Dataset shape after encoding: {df_encoded.shape}')
    print('✅ One-hot encoding completed!')
else:
    print('❌ No categorical columns found for encoding')
    df_encoded = df_cleaned.copy()


## 10. Normalization/Standardization

Applying normalization and standardization for numerical features.

In [ ]:
# Normalization and Standardization
print('=' * 60)
print('NORMALIZATION AND STANDARDIZATION')
print('=' * 60)

# Get numerical columns from encoded dataframe
numerical_cols_encoded = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
# Exclude target variable if encoded
numerical_cols_encoded = [col for col in numerical_cols_encoded if 'label' not in col.lower()]

print(f'Numerical columns for scaling: {len(numerical_cols_encoded)}')

if numerical_cols_encoded:
    # Create copies for different scaling methods
    df_standardized = df_encoded.copy()
    df_normalized = df_encoded.copy()
    
    # StandardScaler (Z-score normalization)
    scaler_standard = StandardScaler()
    df_standardized[numerical_cols_encoded] = scaler_standard.fit_transform(
        df_encoded[numerical_cols_encoded]
    )
    
    # MinMaxScaler (0-1 normalization)
    scaler_minmax = MinMaxScaler()
    df_normalized[numerical_cols_encoded] = scaler_minmax.fit_transform(
        df_encoded[numerical_cols_encoded]
    )
    
    print('✅ Scaling completed!')
    print(f'Standardized data shape: {df_standardized.shape}')
    print(f'Normalized data shape: {df_normalized.shape}')
    
    # Compare scaling effects
    if len(numerical_cols_encoded) > 0:
        sample_col = numerical_cols_encoded[0]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Original
        axes[0].hist(df_encoded[sample_col], bins=30, alpha=0.7, color='blue')
        axes[0].set_title(f'Original {sample_col}')
        axes[0].set_ylabel('Frequency')
        
        # Standardized
        axes[1].hist(df_standardized[sample_col], bins=30, alpha=0.7, color='green')
        axes[1].set_title(f'Standardized {sample_col}')
        axes[1].set_ylabel('Frequency')
        
        # Normalized
        axes[2].hist(df_normalized[sample_col], bins=30, alpha=0.7, color='red')
        axes[2].set_title(f'Normalized {sample_col}')
        axes[2].set_ylabel('Frequency')
        
        plt.tight_layout()
        plt.suptitle('Comparison of Scaling Methods', y=1.02)
        plt.show()
else:
    print('❌ No numerical columns found for scaling')
    df_standardized = df_encoded.copy()
    df_normalized = df_encoded.copy()


## 11. Balancing Techniques

Applying SMOTE, oversampling, undersampling, and Tomek links for class balancing.

In [ ]:
# Balancing Techniques
print('=' * 60)
print('BALANCING TECHNIQUES')
print('=' * 60)

# Check if we have a target variable
if 'label' in df_standardized.columns:
    X = df_standardized.drop('label', axis=1)
    y = df_standardized['label']
    
    print(f'Original class distribution:')
    print(y.value_counts())
    
    # Check if dataset is imbalanced
    class_counts = y.value_counts()
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f'\nImbalance ratio: {imbalance_ratio:.2f}')
    
    if imbalance_ratio > 1.5:
        print('⚠️  Dataset is imbalanced. Applying balancing techniques...')
        
        # 1. SMOTE
        smote = SMOTE(random_state=42)
        X_smote, y_smote = smote.fit_resample(X, y)
        print(f'SMOTE - New shape: {X_smote.shape}')
        
        # 2. Random Oversampling
        ros = RandomOverSampler(random_state=42)
        X_ros, y_ros = ros.fit_resample(X, y)
        print(f'Random Oversampling - New shape: {X_ros.shape}')
        
        # 3. Random Undersampling
        rus = RandomUnderSampler(random_state=42)
        X_rus, y_rus = rus.fit_resample(X, y)
        print(f'Random Undersampling - New shape: {X_rus.shape}')
        
        # 4. Tomek Links
        tomek = TomekLinks()
        X_tomek, y_tomek = tomek.fit_resample(X, y)
        print(f'Tomek Links - New shape: {X_tomek.shape}')
        
        # Use SMOTE data for further analysis
        X_balanced, y_balanced = X_smote, y_smote
        print('✅ Using SMOTE-balanced data for further analysis')
    else:
        print('✅ Dataset is already balanced!')
        X_balanced, y_balanced = X, y
else:
    print('❌ No target variable found for balancing')
    X_balanced = df_standardized
    y_balanced = None


## 12. Correlation Matrix Analysis

Analyzing correlations and feature selection based on correlation matrix.

In [ ]:
# Correlation Matrix Analysis
print('=' * 60)
print('CORRELATION MATRIX ANALYSIS')
print('=' * 60)

# Get numerical columns for correlation analysis
if isinstance(X_balanced, pd.DataFrame):
    numerical_features = X_balanced.select_dtypes(include=[np.number]).columns.tolist()
else:
    # Convert to DataFrame if it's a numpy array
    X_balanced = pd.DataFrame(X_balanced)
    numerical_features = X_balanced.columns.tolist()

print(f'Number of numerical features for correlation: {len(numerical_features)}')

if len(numerical_features) > 1:
    # Calculate correlation matrix
    corr_matrix = X_balanced[numerical_features].corr()
    
    # 1. Pearson Correlation Heatmap
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": .8})
    plt.title('Pearson Correlation Matrix (Upper Triangle)')
    plt.tight_layout()
    plt.show()
    
    # 2. Find highly correlated features
    high_corr_threshold = 0.8
    high_corr_pairs = []
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
                high_corr_pairs.append({
                    'feature1': corr_matrix.columns[i],
                    'feature2': corr_matrix.columns[j],
                    'correlation': corr_matrix.iloc[i, j]
                })
    
    print(f'\nHighly correlated feature pairs (|r| > {high_corr_threshold}):')
    if high_corr_pairs:
        for pair in high_corr_pairs:
            print(f"  {pair['feature1']} <-> {pair['feature2']}: {pair['correlation']:.3f}")
        
        # Features to drop (keep first occurrence)
        features_to_drop = []
        for pair in high_corr_pairs:
            if pair['feature2'] not in features_to_drop:
                features_to_drop.append(pair['feature2'])
        
        print(f'\nFeatures recommended for removal: {features_to_drop}')
        
        # Create dataset with dropped features
        X_corr_filtered = X_balanced.drop(columns=features_to_drop)
        print(f'Shape after correlation-based feature removal: {X_corr_filtered.shape}')
    else:
        print('  No highly correlated features found')
        X_corr_filtered = X_balanced.copy()
    
    # 3. Spearman Correlation (for non-linear relationships)
    spearman_corr = X_balanced[numerical_features].corr(method='spearman')
    
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(spearman_corr, dtype=bool))
    sns.heatmap(spearman_corr, mask=mask, annot=True, cmap='viridis', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": .8})
    plt.title('Spearman Correlation Matrix (Upper Triangle)')
    plt.tight_layout()
    plt.show()
    
    print('✅ Correlation analysis completed!')
else:
    print('❌ Not enough numerical features for correlation analysis')
    X_corr_filtered = X_balanced.copy()


## 13. Outlier Detection and Removal

Detecting and removing outliers using multiple techniques.

In [ ]:
# Outlier Detection and Removal
print('=' * 60)
print('OUTLIER DETECTION AND REMOVAL')
print('=' * 60)

# Get numerical columns for outlier detection
numerical_cols_outlier = X_corr_filtered.select_dtypes(include=[np.number]).columns.tolist()

print(f'Numerical columns for outlier detection: {len(numerical_cols_outlier)}')

if len(numerical_cols_outlier) > 0:
    # 1. IQR Method
    print('\n1. IQR (Interquartile Range) Method')
    outliers_iqr = {}
    X_no_outliers_iqr = X_corr_filtered.copy()
    
    for col in numerical_cols_outlier[:5]:  # Limit to first 5 for performance
        Q1 = X_corr_filtered[col].quantile(0.25)
        Q3 = X_corr_filtered[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = X_corr_filtered[(X_corr_filtered[col] < lower_bound) | 
                                  (X_corr_filtered[col] > upper_bound)].index
        outliers_iqr[col] = len(outliers)
        print(f'  {col}: {len(outliers)} outliers detected')
    
    # 2. Z-Score Method
    print('\n2. Z-Score Method (|z| > 3)')
    outliers_zscore = {}
    
    for col in numerical_cols_outlier[:5]:
        z_scores = np.abs(stats.zscore(X_corr_filtered[col]))
        outliers = X_corr_filtered[z_scores > 3].index
        outliers_zscore[col] = len(outliers)
        print(f'  {col}: {len(outliers)} outliers detected')
    
    # 3. Isolation Forest
    print('\n3. Isolation Forest Method')
    iso_forest = IsolationForest(contamination=0.1, random_state=42)
    outlier_labels = iso_forest.fit_predict(X_corr_filtered[numerical_cols_outlier[:10]])
    outliers_iso = np.sum(outlier_labels == -1)
    print(f'  Isolation Forest: {outliers_iso} outliers detected')
    
    # 4. Local Outlier Factor
    print('\n4. Local Outlier Factor (LOF)')
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
    outlier_labels_lof = lof.fit_predict(X_corr_filtered[numerical_cols_outlier[:10]])
    outliers_lof = np.sum(outlier_labels_lof == -1)
    print(f'  LOF: {outliers_lof} outliers detected')
    
    # Visualization of outliers (using first numerical column)
    if len(numerical_cols_outlier) > 0:
        sample_col = numerical_cols_outlier[0]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Box plot
        axes[0,0].boxplot(X_corr_filtered[sample_col])
        axes[0,0].set_title(f'Box Plot - {sample_col}')
        axes[0,0].set_ylabel('Values')
        
        # Histogram
        axes[0,1].hist(X_corr_filtered[sample_col], bins=30, alpha=0.7)
        axes[0,1].set_title(f'Histogram - {sample_col}')
        axes[0,1].set_xlabel('Values')
        axes[0,1].set_ylabel('Frequency')
        
        # Scatter plot with outliers highlighted
        normal_mask = outlier_labels == 1
        outlier_mask = outlier_labels == -1
        
        axes[1,0].scatter(range(len(X_corr_filtered)), X_corr_filtered[sample_col], 
                         c='blue', alpha=0.6, label='Normal')
        if np.any(outlier_mask):
            outlier_indices = X_corr_filtered.index[outlier_mask]
            axes[1,0].scatter(outlier_indices, X_corr_filtered.loc[outlier_indices, sample_col], 
                             c='red', alpha=0.8, label='Outliers')
        axes[1,0].set_title(f'Outliers Detection - {sample_col}')
        axes[1,0].set_xlabel('Index')
        axes[1,0].set_ylabel('Values')
        axes[1,0].legend()
        
        # Q-Q plot
        stats.probplot(X_corr_filtered[sample_col], dist="norm", plot=axes[1,1])
        axes[1,1].set_title(f'Q-Q Plot - {sample_col}')
        
        plt.tight_layout()
        plt.show()
    
    # Remove outliers using Isolation Forest
    X_final = X_corr_filtered[outlier_labels == 1].copy()
    if y_balanced is not None:
        y_final = y_balanced[outlier_labels == 1]
    else:
        y_final = None
    
    print(f'\nFinal dataset shape after outlier removal: {X_final.shape}')
    print(f'Removed {len(X_corr_filtered) - len(X_final)} outliers')
    print('✅ Outlier detection and removal completed!')
else:
    print('❌ No numerical columns found for outlier detection')
    X_final = X_corr_filtered.copy()
    y_final = y_balanced


## 14. Machine Learning Models Comparison

Applying at least 5 ML models and creating a comparison table.

In [ ]:
# Machine Learning Models Comparison
print('=' * 60)
print('MACHINE LEARNING MODELS COMPARISON')
print('=' * 60)

if y_final is not None and len(X_final) > 0:
    # Prepare data for modeling
    X_train, X_test, y_train, y_test = train_test_split(
        X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
    )
    
    print(f'Training set shape: {X_train.shape}')
    print(f'Test set shape: {X_test.shape}')
    
    # Define models
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'SVM': SVC(random_state=42, probability=True),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
        'Naive Bayes': GaussianNB()
    }
    
    # Results storage
    results = []
    
    print('\nTraining and evaluating models...')
    
    for name, model in models.items():
        print(f'\nTraining {name}...')
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        
        # Train model
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # ROC AUC (for binary classification)
        try:
            if len(np.unique(y_test)) == 2 and y_pred_proba is not None:
                roc_auc = roc_auc_score(y_test, y_pred_proba)
            else:
                roc_auc = roc_auc_score(y_test, y_pred, multi_class='ovr', average='weighted')
        except:
            roc_auc = np.nan
        
        # Store results
        results.append({
            'Model': name,
            'CV_Mean_Accuracy': cv_scores.mean(),
            'CV_Std_Accuracy': cv_scores.std(),
            'Test_Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1_Score': f1,
            'ROC_AUC': roc_auc,
            'Training_Time': training_time
        })
        
        print(f'  ✅ {name} completed - Accuracy: {accuracy:.4f}')
    
    # Create comparison DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.round(4)
    
    print('\n' + '=' * 80)
    print('MODEL COMPARISON TABLE')
    print('=' * 80)
    print(results_df.to_string(index=False))
    
    # Sort by test accuracy
    results_df_sorted = results_df.sort_values('Test_Accuracy', ascending=False)
    
    print('\n🏆 TOP 3 MODELS BY TEST ACCURACY:')
    for i, (_, row) in enumerate(results_df_sorted.head(3).iterrows(), 1):
        print(f'{i}. {row["Model"]}: {row["Test_Accuracy"]:.4f}')
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Accuracy comparison
    axes[0,0].bar(results_df['Model'], results_df['Test_Accuracy'], color='skyblue')
    axes[0,0].set_title('Test Accuracy Comparison')
    axes[0,0].set_ylabel('Accuracy')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. F1 Score comparison
    axes[0,1].bar(results_df['Model'], results_df['F1_Score'], color='lightgreen')
    axes[0,1].set_title('F1 Score Comparison')
    axes[0,1].set_ylabel('F1 Score')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. Training time comparison
    axes[1,0].bar(results_df['Model'], results_df['Training_Time'], color='lightcoral')
    axes[1,0].set_title('Training Time Comparison')
    axes[1,0].set_ylabel('Time (seconds)')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 4. ROC AUC comparison (if available)
    valid_roc = results_df.dropna(subset=['ROC_AUC'])
    if not valid_roc.empty:
        axes[1,1].bar(valid_roc['Model'], valid_roc['ROC_AUC'], color='gold')
        axes[1,1].set_title('ROC AUC Comparison')
        axes[1,1].set_ylabel('ROC AUC')
        axes[1,1].tick_params(axis='x', rotation=45)
    else:
        axes[1,1].text(0.5, 0.5, 'ROC AUC not available', ha='center', va='center')
        axes[1,1].set_title('ROC AUC Comparison')
    
    plt.tight_layout()
    plt.show()
    
    # Save results
    results_df.to_csv('eda/model_comparison_results.csv', index=False)
    print('\n💾 Results saved to: eda/model_comparison_results.csv')
    
    print('\n✅ Machine Learning models comparison completed!')
else:
    print('❌ No target variable or data available for modeling')


## 🎉 Comprehensive Analysis Complete!

### 📋 Summary of Analysis:

1. **📊 Data Loading & Insights**: Loaded Jigsaw Toxic Comment dataset with comprehensive profiling
2. **📈 Visualization**: Created histograms, bar charts, pair plots, and relationship analysis
3. **🔧 Data Preprocessing**: Handled missing data with appropriate strategies
4. **⚙️ Feature Engineering**: Applied stratified sampling, one-hot encoding, and normalization
5. **⚖️ Data Balancing**: Used SMOTE, oversampling, undersampling, and Tomek links
6. **🔗 Correlation Analysis**: Identified and removed highly correlated features
7. **🎯 Outlier Detection**: Applied multiple outlier detection techniques
8. **🤖 Machine Learning**: Compared multiple ML models with comprehensive metrics

### 🚀 Next Steps for Kaggle Competition:

1. **Model Selection**: Choose the best performing model from the comparison
2. **Hyperparameter Tuning**: Fine-tune the selected model parameters
3. **Feature Engineering**: Create additional text-based features (TF-IDF, word embeddings)
4. **Ensemble Methods**: Combine multiple models for better performance
5. **Cross-Validation**: Implement robust validation strategy
6. **Submission**: Generate predictions for test set and create submission file

### 💡 Key Insights for Toxic Comment Classification:

- **Class Imbalance**: The dataset shows significant class imbalance - consider using appropriate sampling techniques
- **Text Features**: Comment length, special characters, and word patterns are important indicators
- **Multi-label Nature**: Comments can have multiple types of toxicity simultaneously
- **Model Performance**: Tree-based models and linear models show promising results for this task

### 🔗 Useful Resources:

- [Jigsaw Toxic Comment Classification Challenge](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge)
- [Evaluation Metric: ROC AUC](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/overview/evaluation)
- [Discussion Forum](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/discussion)

---

**📝 Note**: This notebook provides a comprehensive foundation for the Jigsaw Toxic Comment Classification challenge. Feel free to fork, modify, and extend this analysis for your own submissions!

### Key Files Generated:
- `model_comparison_results.csv`: Detailed comparison of all ML models

### Next Steps:
- Fine-tune the best performing models
- Perform feature selection based on model importance
- Consider ensemble methods for improved performance
- Deploy the best model for production use